<a href="https://colab.research.google.com/github/Startclass15/ProgramacionAvanzadaA-I-2026/blob/main/Proyecto_de_programacion_avanzada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import cv2
import os
import uuid
import time
import nest_asyncio
import asyncio

from telegram import Update
from telegram.ext import ApplicationBuilder, MessageHandler, filters, ContextTypes

# 🔴 TOKEN
TOKEN = "8456187371:AAEfknCDx6ai-jkrjvnPcTJbpz0ARlvtVjs"

# 🔥 SOLUCION AL ERROR DEL EVENT LOOP
nest_asyncio.apply()

# Clasificador Haar
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def procesarImagen(rutaEntrada, rutaSalida):
    imagen = cv2.imread(rutaEntrada)

    if imagen is None:
        print("❌ Error al leer la imagen")
        return False

    grises = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
    rostros = face_cascade.detectMultiScale(grises, 1.5, 8)

    for (x, y, w, h) in rostros:
        cv2.rectangle(imagen, (x, y), (x+w, y+h), (0,255,0), 2)

    cv2.imwrite(rutaSalida, imagen)

    # 🔥 MENSAJE EN CONSOLA
    print("✅ Imagen procesada")

    return True


async def obtenerImagen(update: Update, context: ContextTypes.DEFAULT_TYPE):
    try:
        if not update.message or not update.message.photo:
            return

        foto = update.message.photo[-1]
        archivo = await foto.get_file()

        nombre = str(uuid.uuid4())
        rutaEntrada = f"entrada_{nombre}.jpg"
        rutaSalida = f"salida_{nombre}.jpg"

        await archivo.download_to_drive(rutaEntrada)

        procesarImagen(rutaEntrada, rutaSalida)

        time.sleep(0.5)

        if os.path.exists(rutaSalida):
            with open(rutaSalida, "rb") as img:
                await update.message.reply_photo(photo=img)
        else:
            await update.message.reply_text("No se detectaron rostros")

        os.remove(rutaEntrada)
        os.remove(rutaSalida)

    except Exception as e:
        print("❌ Error:", e)
        await update.message.reply_text("Error al procesar")


async def main():
    app = ApplicationBuilder().token(TOKEN).build()

    app.add_handler(MessageHandler(filters.PHOTO, obtenerImagen))

    # 🔥 MENSAJE CUANDO EL BOT YA ESTÁ FUNCIONANDO
    print("🤖 Bot funcionando...")

    await app.initialize()
    await app.start()
    await app.updater.start_polling()
    await asyncio.Event().wait()


# Ejecutar en Colab
asyncio.get_event_loop().run_until_complete(main())

ModuleNotFoundError: No module named 'telegram'